# 흥행 등급 클러스터링 (K-Means)

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import ast

# 데이터 로드
df = pd.read_csv('steam_indie_9692.csv')

# 윌슨 스코어 파생 변수 추가 (한 번만 계산)
def wilson_score(positive, negative):
    n = positive + negative
    if n == 0: return 0
    z = 1.96 
    phat = positive / n
    return (phat + z**2/(2*n) - z * np.sqrt((phat*(1-phat)+z**2/(4*n))/n)) / (1+z**2/n)

if 'wilson_score' not in df.columns:
    df['wilson_score'] = df.apply(lambda x: wilson_score(x['positive'], x['negative']), axis=1)

# 공통 클러스터링 함수
def perform_tier_clustering(data, title):
    analysis_df = data[['appid', 'name', 'total_reviews']].copy()
    analysis_df['log_reviews'] = np.log1p(analysis_df['total_reviews'])
    
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    analysis_df['cluster'] = kmeans.fit_predict(analysis_df[['log_reviews']])
    
    stats = analysis_df.groupby('cluster')['total_reviews'].agg(['min', 'max', 'count', 'median']).reset_index()
    stats = stats.sort_values(by='min', ascending=False).reset_index(drop=True)
    stats['등급'] = ['Tier 1 (Mega)', 'Tier 2 (Major)', 'Tier 3 (Solid)', 'Tier 4 (Struggling)']
    
    print(f"\n{'='*60}")
    print(f" {title}")
    print(f"{'='*60}")
    print(stats[['등급', 'min', 'max', 'median', 'count']].to_string(index=False))
    return stats

# [핵심] 미래 호환성: 장르 필터링 유틸리티 함수
def filter_by_genre(data, genre_col, target_genre):
    """데이터가 단일 문자열이든 리스트 형태의 문자열이든 유연하게 필터링합니다."""
    def check_genre(val):
        if pd.isna(val): return False
        # 단일 문자열인 경우 (예: "Action")
        if isinstance(val, str) and not val.startswith('['):
            return target_genre == val or target_genre in val.split(',')
        # 리스트 형태의 문자열인 경우 (예: "['Action', 'RPG']")
        try:
            val_list = ast.literal_eval(val) if isinstance(val, str) else val
            if isinstance(val_list, list):
                return target_genre in val_list
        except:
            return target_genre in str(val)
        return False
    
    return data[data[genre_col].apply(check_genre)].copy()

### [버전 1] 윌슨 스코어 미적용 & 전체 게임

In [3]:
title_v1 = f"[Version 1] 전체 게임 흥행 등급 (Wilson 미적용, N={len(df)})"
stats_v1 = perform_tier_clustering(df, title_v1)
display(stats_v1)


 [Version 1] 전체 게임 흥행 등급 (Wilson 미적용, N=9692)
                 등급  min    max  median  count
      Tier 1 (Mega) 1256 370046  3038.0    631
     Tier 2 (Major)  156   1252   347.0   1770
     Tier 3 (Solid)   34    155    63.0   2979
Tier 4 (Struggling)   10     33    17.0   4312


,cluster,min,max,count,median,등급
0,1,1256,370046,631,3038.0,Tier 1 (Mega)
1,3,156,1252,1770,347.0,Tier 2 (Major)
2,2,34,155,2979,63.0,Tier 3 (Solid)
3,0,10,33,4312,17.0,Tier 4 (Struggling)


### [버전 2] 윌슨 스코어 미적용 & 장르별 게임
분석할 장르 컬럼 이름 (나중에 'main_genre' 등으로 변경되면 여기만 바꾸면 됨)

In [ ]:
GENRE_COLUMN = 'genres'

# (선택) 고정 장르 대신, 상위 5개 장르 자동 추출 로직도 가능하지만, 
# 일관성을 위해 기존 5대 장르로 진행합니다. 추후 변경 가능합니다.
target_genres = ['Action', 'Adventure', 'RPG', 'Strategy', 'Simulation']

print("\n### [Version 2] 주요 장르별 흥행 등급 (Wilson 미적용) ###")
for genre in target_genres:
    # 개선된 유틸리티 함수 사용
    genre_df = filter_by_genre(df, GENRE_COLUMN, genre)
    
    if not genre_df.empty and len(genre_df) > 50: # 최소 샘플 수 방어 로직 추가
        title_v2 = f"장르: {genre} (N={len(genre_df)})"
        perform_tier_clustering(genre_df, title_v2)


### [Version 2] 주요 장르별 흥행 등급 (Wilson 미적용) ###

 장르: Action (N=4093)
                 등급  min    max  median  count
      Tier 1 (Mega) 1879 370046  5235.5    232
     Tier 2 (Major)  198   1846   444.0    691
     Tier 3 (Solid)   39    196    73.0   1115
Tier 4 (Struggling)   10     38    18.0   2055

 장르: Adventure (N=4807)
                 등급  min    max  median  count
      Tier 1 (Mega) 1483 370046  3863.0    311
     Tier 2 (Major)  178   1478   385.5    844
     Tier 3 (Solid)   38    177    70.0   1424
Tier 4 (Struggling)   10     37    18.0   2228

 장르: RPG (N=2194)
                 등급  min    max  median  count
      Tier 1 (Mega) 1573 370046  3970.5    198
     Tier 2 (Major)  199   1549   412.5    502
     Tier 3 (Solid)   42    198    84.5    632
Tier 4 (Struggling)   10     41    19.0    862

 장르: Strategy (N=2005)
                 등급  min    max  median  count
      Tier 1 (Mega) 2140 370046  5957.0    115
     Tier 2 (Major)  231   2004   519.0    405
     Tier 3 (Soli

### [버전 3] 윌슨 스코어 적용 & 전체 게임

In [7]:
score_threshold_total = df['wilson_score'].quantile(0.75)
df_high_total = df[df['wilson_score'] >= score_threshold_total].copy()

title_v3 = f"[Version 3] 민심 검증 전체 게임 (Wilson >= {score_threshold_total:.3f}, N={len(df_high_total)})"
stats_v3 = perform_tier_clustering(df_high_total, title_v3)


 [Version 3] 민심 검증 전체 게임 (Wilson >= 0.839, N=2460)
                 등급  min    max  median  count
      Tier 1 (Mega) 4209 370046  9970.0    190
     Tier 2 (Major)  549   4129  1275.0    473
     Tier 3 (Solid)   98    546   224.0    768
Tier 4 (Struggling)   20     97    43.0   1029


### [버전 4] 윌슨 스코어 적용 & 장르별 게임

In [ ]:
GENRE_COLUMN = 'genres'
print("\n### [Version 4] 주요 장르별 민심 검증 흥행 등급 (Wilson Score)###")

for genre in target_genres:
    # 개선된 유틸리티 함수 사용
    genre_df = filter_by_genre(df, GENRE_COLUMN, genre)
    
    if not genre_df.empty and len(genre_df) > 50:
        genre_threshold = genre_df['wilson_score'].quantile(0.75)
        genre_high_df = genre_df[genre_df['wilson_score'] >= genre_threshold].copy()
        
        title_v4 = f"장르: {genre} (Wilson >= {genre_threshold:.3f}, N={len(genre_high_df)})"
        perform_tier_clustering(genre_high_df, title_v4)


### [Version 4] 주요 장르별 민심 검증 흥행 등급 (Wilson Score)###

 장르: Action (Wilson >= 0.832, N=1044)
                 등급  min    max  median  count
      Tier 1 (Mega) 4289 370046 11496.0     99
     Tier 2 (Major)  488   4101  1197.5    218
     Tier 3 (Solid)   83    472   179.0    310
Tier 4 (Struggling)   19     81    35.0    417

 장르: Adventure (Wilson >= 0.839, N=1203)
                 등급  min    max  median  count
      Tier 1 (Mega) 4771 370046 10597.5    104
     Tier 2 (Major)  628   4678  1373.5    226
     Tier 3 (Solid)  109    611   261.0    361
Tier 4 (Struggling)   20    107    47.0    512

 장르: RPG (Wilson >= 0.839, N=549)
                 등급  min    max  median  count
      Tier 1 (Mega) 6249 370046 13336.0     49
     Tier 2 (Major)  822   5957  1885.0    111
     Tier 3 (Solid)  126    784   311.0    199
Tier 4 (Struggling)   20    123    49.5    190

 장르: Strategy (Wilson >= 0.832, N=502)
                 등급  min    max  median  count
      Tier 1 (Mega) 5307 370046 11550.

: 